# Bangla Hallucination Detection - Model Pipeline (v1.0)

This notebook trains a Bangla LLM Hallucination Detection system designed to run on Kaggle.
It implements a robust character-level TF-IDF model coupled with feature extraction and a Logistic Regression classifier, optimized using 5-fold cross-validation.

### Version History
- **v1.0 (Current)**: Initial baseline version using character-level n-gram TF-IDF on `prompt_bn` and `response_bn` and robust data cleaning.

In [ ]:
import json
import numpy as np
import pandas as pd
import os
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import FeatureUnion, Pipeline
from sklearn.preprocessing import FunctionTransformer
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, classification_report, confusion_matrix

RANDOM_STATE = 42
DATA_PATH = "/kaggle/input/competitions/bengali-hallucination/dataset samples.json"
TEST_PATH = "/kaggle/input/competitions/bengali-hallucination/test set.csv"
SUBMISSION_PATH = "submission.csv"

## 1. Load and Clean Data

In [ ]:
# Determine if we are running in Kaggle environment or locally
if not os.path.exists(DATA_PATH):
    # Local fallback paths
    DATA_PATH = "../bengali-hallucination/dataset samples.json"
    TEST_PATH = "../bengali-hallucination/test set.csv"

print(f"Using DATA_PATH: {DATA_PATH}")
print(f"Using TEST_PATH: {TEST_PATH}")

with open(DATA_PATH, encoding="utf-8") as f:
    records = json.load(f)

df = pd.DataFrame(records)
print(f"{len(df)} training rows loaded.")

# Clean text columns
for col in ["prompt_bn", "response_bn"]:
    df[col] = df[col].astype(str)

NO_CONTEXT_VALUES = {"", "nan", "NaN", "[NULL]", None}

def clean_context(value):
    if pd.isna(value) or str(value).strip() in NO_CONTEXT_VALUES:
        return ""
    return str(value).strip()

df["context"] = df["context"].apply(clean_context)
df["has_context"] = df["context"].str.len() > 0

## 2. Define Model Pipeline

In [ ]:
def select_column(name):
    return FunctionTransformer(lambda frame: frame[name].to_numpy(), validate=False)

def tfidf_branch(name):
    return Pipeline([
        ("select", select_column(name)),
        ("tfidf", TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 4), max_features=20000)),
    ])

model = Pipeline([
    ("features", FeatureUnion([
        ("prompt", tfidf_branch("prompt_bn")),
        ("response", tfidf_branch("response_bn")),
    ])),
    ("clf", LogisticRegression(max_iter=1000, class_weight="balanced", random_state=RANDOM_STATE)),
])

## 3. Train & Evaluate (5-Fold CV)

In [ ]:
y = df["label"].values
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

cv_pred = cross_val_predict(model, df, y, cv=cv, method="predict")
cv_proba = cross_val_predict(model, df, y, cv=cv, method="predict_proba")[:, 1]

print("=== v1.0 Model Stratified 5-Fold Cross-Validation Metrics ===")
print(f"Accuracy : {accuracy_score(y, cv_pred):.4f}")
print(f"Macro-F1 : {f1_score(y, cv_pred, average='macro'):.4f} (Primary Metric)")
print(f"F1 (label=1 - faithful): {f1_score(y, cv_pred, pos_label=1):.4f}")
print(f"F1 (label=0 - hallucinated): {f1_score(y, cv_pred, pos_label=0):.4f}")
print(f"AUC-ROC  : {roc_auc_score(y, cv_proba):.4f}")
print("\nConfusion Matrix:")
print(confusion_matrix(y, cv_pred))

## 4. Train on Full Dataset and Predict on Test Set

In [ ]:
model.fit(df, y)
print("Model trained on full dataset.")

if os.path.exists(TEST_PATH):
    test_df = pd.read_csv(TEST_PATH)
    print(f"Loaded test set with {len(test_df)} rows.")
    for col in ["prompt_bn", "response_bn"]:
        test_df[col] = test_df[col].astype(str)
    test_df["context"] = test_df["context"].apply(clean_context)
    
    preds = model.predict(test_df)
    
    submission = pd.DataFrame({
        "id": test_df["id"] if "id" in test_df.columns else range(len(test_df)),
        "label": preds
    })
    submission.to_csv(SUBMISSION_PATH, index=False)
    print(f"Submission file saved to {SUBMISSION_PATH}. Rows: {len(submission)}")
else:
    print("Test set CSV file not found. Skipping prediction. Model is ready for Kaggle environment execution.")